# Friends Demo — Phase 1: Core Features & Lifecycle

Walkthrough of Mem0's core capabilities: **Store → Search → Scope → Update → Delete → Metadata Filters**.




A single walkthrough of mem0's core, everyday capabilities: **store → search → scope by
user → update → delete → filter → multimodal**.

Different example from the aircraft engineer notebooks on purpose -- this one uses a small
group of friends with everyday interests, so the facts and queries are easy to follow at a
glance:

- **Maya** -- likes hiking, is learning pottery
- **Jordan** -- likes cooking, plays jazz piano
- **Sam** -- into video games, allergic to peanuts

Same `MemoryClient` setup as the other notebooks.


### 📋 Phase 1 Overview & Execution Flow

Phase 1 demonstrates the core lifecycle operations:

- **Step 1 — Store (`client.add`)**: Automatic fact extraction from raw conversational text for Maya, Jordan, and Sam.
- **Step 2 — Semantic Search (`client.search`)**: Search user memories by meaning rather than exact keyword matches.
- **Step 3 — User Isolation Scope**: Demonstrates strict memory segregation per `user_id` (`maya`, `jordan`, `sam`).
- **Step 4 — Update Memory (`client.update`)**: Updates a specific memory in-place via `memory_id`.
- **Step 5 — Delete Memory (`client.delete`)**: Deletes a specific memory via `memory_id` and verifies removal.
- **Step 6 — Metadata Filtering**: Attach custom metadata tags (e.g. `topic: cooking`) for logical querying.
- **Step 7 — Multimodal Context**: Overview of image URL fact extraction.


## Setup

In [8]:
import os
from dotenv import load_dotenv
from mem0 import MemoryClient

load_dotenv()
client = MemoryClient(api_key=os.getenv("MEM0_API_KEY"))
print("Connected to mem0.")

Connected to mem0.


In [ ]:
client.delete_all(user_id="maya")
client.delete_all(user_id="sam")
client.delete_all(user_id="jordan")

## Step 1 — Add conversations for a few different people

We give mem0 raw sentences, the same way a real conversation would sound. mem0's LLM
extracts the facts worth remembering -- we're not writing the facts ourselves.


In [9]:
conversations = [
    ("maya",   "I went hiking in the Rockies last weekend, it was amazing. I've also just started a pottery class on Tuesdays."),
    ("jordan", "I spent the whole afternoon cooking a big Italian dinner for friends. I also play jazz piano most evenings to unwind."),
    ("sam",    "Got really into this new video game this week. Also, heads up, I'm allergic to peanuts, so no peanut sauce next time."),
]

for user_id, message in conversations:
    result = client.add(message, user_id=user_id)
    print(f"[{user_id}] add() returned:", result)

[maya] add() returned: {'event_id': '0ae99601-502e-4c95-8ad4-1f3e9b2cf0fd', 'status': 'PENDING'}
[jordan] add() returned: {'event_id': '0aac3600-7295-43be-866e-f44d697aab9c', 'status': 'PENDING'}
[sam] add() returned: {'event_id': '22fc52ef-c159-4ffd-b371-2be8e2dc83d9', 'status': 'PENDING'}


In [10]:
conversations = [
    ("maya",   "I went hiking in the Rockies last weekend, it was amazing. I've also just started a pottery class on Tuesdays."),
    ("jordan", "I spent the whole afternoon cooking a big Italian dinner for friends. I also play jazz piano most evenings to unwind."),
    ("sam",    "Got really into this new video game this week. Also, heads up, I'm allergic to peanuts, so no peanut sauce next time."),
]

for user_id, message in conversations:
    result = client.add(message, user_id=user_id, infer=False)
    print(f"[{user_id}] add() returned:", result)

[maya] add() returned: {'message': 'Memories stored successfully', 'status': 'SUCCEEDED', 'event_id': 'dbd40214-ba7b-4865-a418-b8b25307edbb', 'results': [{'id': '75f32def-7616-4423-9bc8-8b01e7a76b3e', 'data': {'memory': "I went hiking in the Rockies last weekend, it was amazing. I've also just started a pottery class on Tuesdays."}, 'event': 'ADD'}]}
[jordan] add() returned: {'message': 'Memories stored successfully', 'status': 'SUCCEEDED', 'event_id': '89b7122a-098d-4369-8cef-e18e9202b6f0', 'results': [{'id': '33076561-7d0d-4b24-94b9-60cefec987be', 'data': {'memory': 'I spent the whole afternoon cooking a big Italian dinner for friends. I also play jazz piano most evenings to unwind.'}, 'event': 'ADD'}]}
[sam] add() returned: {'message': 'Memories stored successfully', 'status': 'SUCCEEDED', 'event_id': '768cc601-3dd9-404f-ae49-db38f9f9dfc0', 'results': [{'id': 'e27364d0-9bae-4346-9200-86eb52512fe7', 'data': {'memory': "Got really into this new video game this week. Also, heads up, I'

### See what actually got extracted for everyone

In [11]:
for user_id in ["maya", "jordan", "sam"]:
    print(f"--- {user_id} ---")
    memories = client.get_all(filters={"user_id": user_id})
    for item in memories.get("results", []):
        print(" -", item["memory"])
    print()

--- maya ---
 - I went hiking in the Rockies last weekend, it was amazing. I've also just started a pottery class on Tuesdays.
 - User recently started attending a pottery class that takes place on Tuesdays
 - User went hiking in the Rockies on the weekend of August 1-2, 2026 and found it amazing

--- jordan ---
 - I spent the whole afternoon cooking a big Italian dinner for friends. I also play jazz piano most evenings to unwind.
 - User plays jazz piano most evenings to unwind
 - User spent the afternoon of August 5, 2026 cooking a big Italian dinner for friends

--- sam ---
 - Got really into this new video game this week. Also, heads up, I'm allergic to peanuts, so no peanut sauce next time.
 - User is allergic to peanuts and requests no peanut sauce in future meals
 - User got really into a new video game during the week of August 5, 2026



## Step 2 — Semantic search

Search by *meaning*, not exact words. Neither of these questions uses the same wording as
what was actually said above.


### "What does Maya like?" -- searching one specific person

In [12]:
results = client.search(query="What does Maya like?", filters={"user_id": "maya"})
for r in results.get("results", []):
    print(f"{r['score']:.3f}  {r['memory']}")

0.193  I went hiking in the Rockies last weekend, it was amazing. I've also just started a pottery class on Tuesdays.
0.166  User recently started attending a pottery class that takes place on Tuesdays
0.148  User went hiking in the Rockies on the weekend of August 1-2, 2026 and found it amazing


### "Who likes hiking?" -- searching across everyone

mem0's v3 API requires at least one entity ID inside `filters` -- it won't search with no
filter at all. To search across our whole friend group, we list everyone explicitly with
`OR`.


In [13]:
results = client.search(
    query="who likes hiking?",
    filters={"OR": [{"user_id": "maya"}, {"user_id": "jordan"}, {"user_id": "sam"}]},
)
for r in results.get("results", []):
    print(f"{r['score']:.3f}  {r['memory']}")
    

0.421  User went hiking in the Rockies on the weekend of August 1-2, 2026 and found it amazing
0.313  I went hiking in the Rockies last weekend, it was amazing. I've also just started a pottery class on Tuesdays.
0.198  I spent the whole afternoon cooking a big Italian dinner for friends. I also play jazz piano most evenings to unwind.
0.197  User plays jazz piano most evenings to unwind
0.193  User spent the afternoon of August 5, 2026 cooking a big Italian dinner for friends
0.193  Got really into this new video game this week. Also, heads up, I'm allergic to peanuts, so no peanut sauce next time.
0.189  User got really into a new video game during the week of August 5, 2026
0.178  User recently started attending a pottery class that takes place on Tuesdays
0.168  User is allergic to peanuts and requests no peanut sauce in future meals


## Step 3 — User-scoped memories stay separate

Same query, three different `user_id` filters. Each person only sees their own facts back,
even though we're asking the exact same question every time.


In [14]:
query = "what are this person's hobbies?"

for user_id in ["maya", "jordan", "sam"]:
    print(f"--- searching as {user_id} ---")
    results = client.search(query=query, filters={"user_id": user_id})
    for r in results.get("results", []):
        print(f"   {r['score']:.3f}  {r['memory']}")
    print()

--- searching as maya ---
   0.294  I went hiking in the Rockies last weekend, it was amazing. I've also just started a pottery class on Tuesdays.
   0.229  User recently started attending a pottery class that takes place on Tuesdays
   0.211  User went hiking in the Rockies on the weekend of August 1-2, 2026 and found it amazing

--- searching as jordan ---
   0.290  I spent the whole afternoon cooking a big Italian dinner for friends. I also play jazz piano most evenings to unwind.
   0.229  User plays jazz piano most evenings to unwind
   0.192  User spent the afternoon of August 5, 2026 cooking a big Italian dinner for friends

--- searching as sam ---
   0.214  Got really into this new video game this week. Also, heads up, I'm allergic to peanuts, so no peanut sauce next time.
   0.187  User got really into a new video game during the week of August 5, 2026
   0.156  User is allergic to peanuts and requests no peanut sauce in future meals



In [24]:
query = "what are this person's hobbies?"

for user_id in ["maya", "jordan", "sam"]:
    print(f"--- searching as {user_id} ---")
    results = client.search(query=query, filters={"user_id": "Alice"})
    for r in results.get("results", []):
        print(f"   {r['score']:.3f}  {r['memory']}")
    print()

--- searching as maya ---

--- searching as jordan ---

--- searching as sam ---



## Step 4 — Update a memory

`update()` needs a specific memory's ID, not just a user_id -- so first we find the ID of
the fact we want to change.


In [15]:
# Find Maya's pottery memory and grab its ID
maya_memories = client.get_all(filters={"user_id": "maya"})

pottery_memory = None
for item in maya_memories.get("results", []):
    if "pottery" in item["memory"].lower():
        pottery_memory = item
        break

print("BEFORE update:")
print(" -", pottery_memory["memory"])
print("memory_id:", pottery_memory["id"])

BEFORE update:
 - I went hiking in the Rockies last weekend, it was amazing. I've also just started a pottery class on Tuesdays.
memory_id: 75f32def-7616-4423-9bc8-8b01e7a76b3e


In [16]:
client.update(
    memory_id=pottery_memory["id"],
    text="Maya's pottery class moved from Tuesdays to Thursdays starting next month.",
)

# Fetch it again to confirm the change actually stuck
after_update = client.get_all(filters={"user_id": "maya"})
for item in after_update.get("results", []):
    if item["id"] == pottery_memory["id"]:
        print("AFTER update:")
        print(" -", item["memory"])

AFTER update:
 - Maya's pottery class moved from Tuesdays to Thursdays starting next month.


## Step 5 — Delete a memory

Same idea -- grab the ID first, then delete it, then confirm it's actually gone.


In [17]:
# Find Sam's peanut allergy memory
sam_memories = client.get_all(filters={"user_id": "sam"})

allergy_memory = None
for item in sam_memories.get("results", []):
    if "peanut" in item["memory"].lower():
        allergy_memory = item
        break

print("BEFORE delete:")
for item in sam_memories.get("results", []):
    print(" -", item["memory"])

BEFORE delete:
 - Got really into this new video game this week. Also, heads up, I'm allergic to peanuts, so no peanut sauce next time.
 - User is allergic to peanuts and requests no peanut sauce in future meals
 - User got really into a new video game during the week of August 5, 2026


In [13]:
client.delete(memory_id=allergy_memory["id"])

after_delete = client.get_all(filters={"user_id": "sam"})
print("AFTER delete:")
for item in after_delete.get("results", []):
    print(" -", item["memory"])

still_there = any(item["id"] == allergy_memory["id"] for item in after_delete.get("results", []))
print("\nDeleted successfully" if not still_there else "Still present -- delete did not take effect")

AFTER delete:
 - User became really into a new video game during the week of August 4–10, 2026
 - User is allergic to peanuts and asks that no peanut sauce be used in the future

Deleted successfully


## Step 6 — Advanced filters: metadata

Beyond `user_id`, you can attach your own `metadata` to a memory when you add it, then filter
on it later -- useful when you want more precise queries than plain semantic search gives you.


In [18]:
client.add(
    "Jordan is planning a jazz-themed dinner party next month and wants recipe ideas.",
    user_id="jordan",
    metadata={"topic": "event_planning"},
)

client.add(
    "Jordan mentioned wanting to try a new ramen recipe this weekend.",
    user_id="jordan",
    metadata={"topic": "cooking"},
)

print("Added two memories with different metadata topics.")

Added two memories with different metadata topics.


In [19]:
# Filter for just Jordan's cooking-related memories, not everything
results = client.get_all(
    filters={"AND": [{"user_id": "jordan"}, {"metadata": {"topic": "cooking"}}]}
)
print("topic=cooking only:")
for item in results.get("results", []):
    print(" -", item["memory"])

topic=cooking only:


## Step 7 — Multimodal (optional)

mem0 can also extract facts from an image, not just text. This cell is optional -- skip it if
you don't have an image handy. Replace `image_url` with a real, publicly accessible image URL
to try it.


In [20]:
image_url = "https://upload.wikimedia.org/wikipedia/commons/4/48/Garden_orb_weaver05.jpg?utm_source=en.wikipedia.org&utm_campaign=index&utm_content=original"  # <-- replace with a real image URL

# Snapshot BEFORE, so we can isolate exactly what the image added
before_image = client.get_all(filters={"user_id": "maya"})
before_ids = {item["id"] for item in before_image.get("results", [])}

MemoryClient
add_result = client.add(
    f"Here is a photo from Maya's hike last weekend: {image_url}",
    user_id="maya",
)

# add() usually returns the newly created memory objects directly -- print that first
print("add() returned:")
print(add_result)

add() returned:
{'event_id': '9c4f5d00-982d-43ba-89e8-9a26cd4e7800', 'status': 'PENDING'}


### Confirm it by re-fetching, and isolate just the new facts

Printing `add()`'s return value shows what mem0 says it stored. Diffing `get_all()` before
and after confirms it, and separates out only the facts that came specifically from the image
-- not anything already there from Step 1.


In [21]:
after_image = client.get_all(filters={"user_id": "maya"})

new_facts = [
    item["memory"] for item in after_image.get("results", [])
    if item["id"] not in before_ids
]

print("New facts extracted from the image:")
if new_facts:
    for f in new_facts:
        print(" -", f)
else:
    print("(none -- if you're still using the placeholder image_url, replace it with a real, "
          "publicly accessible image and re-run this cell)")

New facts extracted from the image:
 - Maya went hiking on the weekend of July 31 to August 1, 2026


## Wrap-up

Full lifecycle demonstrated in one notebook:

1. **Store** -- raw conversation in, extracted facts out, automatically
2. **Search** -- retrieval by meaning, not exact wording
3. **Scope** -- the same query returns completely different results depending on whose
   memories you're searching
4. **Update** -- an existing fact can be edited in place given its `memory_id`
5. **Delete** -- a specific fact can be removed outright, confirmed by re-fetching afterward
6. **Filter** -- `metadata` narrows results beyond what semantic search alone would return
7. **Multimodal** -- the same extraction pipeline can work from an image, not just text

One thing worth connecting back to the earlier notebooks: Steps 4 and 5 here call `update()`
and `delete()` **directly and explicitly**, with a known `memory_id`. That's different from
Phase 5, where we tested whether mem0's automatic conflict-detection would call update/delete
*on its own* when it noticed a contradiction -- and found it mostly didn't. This notebook
shows those operations definitely work when you invoke them yourself; Phase 5 showed mem0
doesn't reliably reach for them on its own. Both things can be true at once, and it's a good
point to make explicitly in the demo.
